In [1]:
import json
from collections import defaultdict
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm

In [2]:
with open("./v1.5/dev/archehr-qa_key.json") as f:
    key = json.load(f)


tree = ET.parse("./v1.5/dev/archehr-qa.xml")
root = tree.getroot()

data = []
for case in root.findall("case"):
    case_id = case.get('id')
    c_question = case.findtext("clinician_question", default="").strip()
    p_question = case.findtext("patient_narrative", default="").strip()
    sentences = [
        {
            'sentence_id': sentence.get('id'),
            'text': sentence.text.strip()
        }
        for sentence in case.find("note_excerpt_sentences")
    ]
    item = {
        'case_id': case_id,
        'excerpt_sentences': sentences,
        'clinician_question': c_question,
        'patient_question': p_question,
    }
    data.append(item)

df1 = pd.DataFrame(data)
df2 = pd.DataFrame(key)
combined = df1.merge(df2[['case_id', 'clinician_answer_sentences']], on='case_id').to_dict(orient='records')

In [3]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
SYSTEM_PROMPT_1 = """You are a clinical evidence alignment system.

Your task is to align answer sentences to supporting clinical note sentences.

Rules:
1. Alignment is at the answer sentence level.
2. Only cite note sentences that DIRECTLY support the answer sentence.
3. Do NOT over-cite.
4. Do NOT under-cite.
5. If no supporting evidence exists, return an empty list.
6. Use only the provided numbered note sentences.
7. Do NOT invent evidence.
8. Do NOT explain your reasoning.
9. Output structured alignment only in the format:

Answer Sentence <ID>:
Supported by Note Sentence(s): <comma separated list OR NONE>

Example:
Answer Sentence 1:
Supported by Note Sentence(s): 3,5

Answer Sentence 2:
Supported by Note Sentence(s): NONE"""

FEWSHOT = """Example:

Patient Question:
I had severe abdomen pain and was hospitalised for 15 days in ICU, diagnoised with CBD sludge. Doctor advised for ERCP. My question is if the sludge was there does not any medication help in flushing it out? Whether ERCP was the only cure?

Clinician-Interpreted Question:
Why was ERCP recommended over a medication-based treatment for CBD sludge?

Clinical Note Excerpt:
1: During the ERCP a pancreatic stent was required to facilitate access to the biliary system (removed at the end of the procedure), and a common bile duct stent was placed to allow drainage of the biliary obstruction caused by stones and sludge.
2: However, due to the patient’s elevated INR, no sphincterotomy or stone removal was performed.
3: Frank pus was noted to be draining from the common bile duct, and post-ERCP it was recommended that the patient remain on IV Zosyn for at least a week.
4: The Vancomycin was discontinued.
5: On hospital day 4 (post-procedure day 3) the patient returned to ERCP for re-evaluation of her biliary stent as her LFTs and bilirubin continued an upward trend.
6: On ERCP the previous biliary stent was noted to be acutely obstructed by biliary sludge and stones.
7: As the patient’s INR was normalized to 1.2, a sphincterotomy was safely performed, with removal of several biliary stones in addition to the common bile duct stent.
8: At the conclusion of the procedure, retrograde cholangiogram was negative for filling defects.

Answer:
1: An endoscopic retrograde cholangiopancreatography, ERCP, was recommended to place a common bile duct stent.
2: This stent was placed to allow drainage of the biliary obstruction which was caused by stones and sludge.
3: Due to no improvement in liver function, the patient needed a repeat ERCP.
4: The repeat ERCP showed that the biliary stent placed in the first ERCP was obstructed by stones and sludge.
5: The stones and stent were successfully removed during this procedure by performing a sphincterotomy.

Output:

Answer Sentence 1:
Supported by Note Sentence(s): 1

Answer Sentence 2:
Supported by Note Sentence(s): 1

Answer Sentence 3:
Supported by Note Sentence(s): 5

Answer Sentence 4:
Supported by Note Sentence(s): 6

Answer Sentence 5:
Supported by Note Sentence(s): 7"""

SYSTEM_PROMPT_1 = SYSTEM_PROMPT_1 + '\n' + FEWSHOT

USER_PROMPT_1 = """Case ID: {}

Patient Question:
{}

Clinician-Interpreted Question:
{}

Clinical Note Excerpt:
{}

Answer:
{}

Perform answer-to-evidence alignment."""

SYSTEM_PROMPT_2 = """Convert the alignment into STRICT JSON.

Rules:
- Output must be valid JSON.
- Output must be an array with one object.
- Use this schema:

[
  {
    "case_id": "<case_id>",
    "prediction": [
      {
        "answer_id": "<ID>",
        "evidence_id": ["<note_id>", ...]
      }
    ]
  }
]

- Use empty array [] if unsupported.
- Evidence IDs must be strings.
- Do not include any explanation.
- Do not include markdown.
- Do not include extra text."""

USER_PROMPT_2 = """Case ID: {}

Alignment:

{}"""

def stage_1(*args):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_1},
        {"role": "user", "content": USER_PROMPT_1.format(*args)},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

def stage_2(*args):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_2},
        {"role": "user", "content": USER_PROMPT_2.format(*args)},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [5]:
from tqdm.notebook import tqdm
from ast import literal_eval

dev_answers = []
for case in tqdm(combined):
    cid = case['case_id']
    pq = case['patient_question']
    cq = case['clinician_question']
    evidence_sentences = '\n'.join(f"{q['sentence_id']}: {q['text'].replace('\n', ' ')}" for q in case['excerpt_sentences'])
    answer_sentences = '\n'.join(f"{q['id']}: {q['text']}" for q in case['clinician_answer_sentences'])
    
    a = stage_1(cid, pq, cq, evidence_sentences, answer_sentences)
    item = stage_2(cid, a)
    try:
        item = literal_eval(item)
    except:
        item = [{'case_id': cid, 'prediction': []}]
        print(f"Problem with {cid}")
    
    dev_answers.extend(item)

  0%|          | 0/20 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
from scoring_subtask_4 import compute_alignment_scores, get_leaderboard, load_key

key_map = load_key("./v1.5/dev/archehr-qa_key.json")

scores = compute_alignment_scores(dev_answers, key_map)
leaderboard = get_leaderboard(scores)
leaderboard

{'micro_precision': 20.0,
 'micro_recall': 11.76470588235294,
 'micro_f1': 14.814814814814817,
 'macro_precision': np.float64(18.63232323232323),
 'macro_recall': np.float64(13.173973156709728),
 'macro_f1': np.float64(14.907441649001749),
 'overall_score': 14.814814814814817}

### 0 shot
{'micro_precision': 28.30188679245283,
 'micro_recall': 20.361990950226243,
 'micro_f1': 23.684210526315788,
 'macro_precision': np.float64(35.41736918052707),
 'macro_recall': np.float64(27.590639823376396),
 'macro_f1': np.float64(30.146750308515013),
 'overall_score': 23.684210526315788}